In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.max_columns = 160

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
DATA_DIR = PROJECT_ROOT / "data"
FEATURE_FILE = DATA_DIR / "features" / "pjm_load_era5_hourly_2023_0101_0107.parquet"

FEATURE_FILE

# Feature-Load Relationship Diagnostics

Purpose: test which available weather and calendar features are associated with PJM load, and separate descriptive correlation from forecast-usable predictive signal.

Research guardrails:
- Same-hour weather/load correlations are descriptive and may be leaky for forecasting.
- Predictive diagnostics must use features available before the forecast target timestamp.
- The current ERA5 sample is only one week, so results are pipeline checks, not research conclusions.
- Later runs should use longer history and regime-stratified walk-forward validation.

In [ ]:
if not FEATURE_FILE.exists():
    raise FileNotFoundError(f"Run join_load_weather_pjm.ipynb first: {FEATURE_FILE}")

features = pd.read_parquet(FEATURE_FILE)
features["timestamp_utc"] = pd.to_datetime(features["timestamp_utc"], utc=True)
features = features.sort_values("timestamp_utc").reset_index(drop=True)

features.shape, features["timestamp_utc"].min(), features["timestamp_utc"].max()

## Candidate Features

Exclude identifiers, target leakage columns, and source metadata. Keep numeric weather/calendar candidates for first-pass screening.

In [ ]:
target_col = "load_mw"
excluded = {
    "load_mw",
    "timestamp_utc",
    "period_raw",
    "respondent",
    "respondent-name",
    "type",
    "type-name",
    "number",
    "expver",
}

numeric_cols = features.select_dtypes(include=["number", "bool"]).columns.tolist()
candidate_cols = [col for col in numeric_cols if col not in excluded]

screening_frame = (
    features[["timestamp_utc", target_col, *candidate_cols]]
    .dropna(subset=[target_col])
    .copy()
)

pd.Series(
    {
        "rows": len(screening_frame),
        "candidate_features": len(candidate_cols),
        "start_utc": screening_frame["timestamp_utc"].min(),
        "end_utc": screening_frame["timestamp_utc"].max(),
    }
)

## Same-Hour Association

This is useful EDA, but not automatically forecast-valid. Realized ERA5 weather at the target hour is not available before that hour in a real forecasting setup.

In [ ]:
def rank_same_hour_association(
    df: pd.DataFrame, target: str, columns: list[str]
) -> pd.DataFrame:
    rows = []
    for col in columns:
        pair = df[[target, col]].dropna()
        if len(pair) < 3 or pair[col].nunique() < 2:
            continue
        rows.append(
            {
                "feature": col,
                "n": len(pair),
                "pearson_corr": pair[target].corr(pair[col], method="pearson"),
                "spearman_corr": pair[target].corr(pair[col], method="spearman"),
            }
        )
    result = pd.DataFrame(rows)
    if result.empty:
        return result
    result["abs_spearman_corr"] = result["spearman_corr"].abs()
    return result.sort_values("abs_spearman_corr", ascending=False).reset_index(
        drop=True
    )


association_rank = rank_same_hour_association(
    screening_frame, target_col, candidate_cols
)
association_rank.head(20)

In [ ]:
top_features = (
    association_rank.head(8)["feature"].tolist() if len(association_rank) else []
)

if top_features:
    fig, axes = plt.subplots(len(top_features), 1, figsize=(8, 3 * len(top_features)))
    axes = np.atleast_1d(axes)
    for ax, col in zip(axes, top_features):
        ax.scatter(screening_frame[col], screening_frame[target_col], s=12, alpha=0.5)
        ax.set_xlabel(col)
        ax.set_ylabel("load_mw")
        ax.set_title(f"Same-hour load relationship: {col}")
    plt.tight_layout()
    plt.show()

## Lagged Predictive Screening

Create lagged versions of candidate features. These are closer to forecast-usable inputs because they only use prior observed values. This is still a rough diagnostic, not final evaluation.

In [ ]:
lag_hours = [1, 24]
lagged = (
    features[["timestamp_utc", target_col, *candidate_cols]]
    .sort_values("timestamp_utc")
    .copy()
)

lagged_feature_cols = []
for col in candidate_cols:
    for lag in lag_hours:
        lagged_col = f"{col}_lag_{lag}h"
        lagged[lagged_col] = lagged[col].shift(lag)
        lagged_feature_cols.append(lagged_col)

lagged_screen = lagged.dropna(subset=[target_col]).copy()
lagged_association_rank = rank_same_hour_association(
    lagged_screen, target_col, lagged_feature_cols
)
lagged_association_rank.head(20)

## Simple Incremental Baseline Check

This compares naive baselines with a tiny linear model using lagged features. With only one week of data, this is a mechanics check. A valid experiment needs a longer window and walk-forward folds.

In [ ]:
def mae(actual: pd.Series, predicted: pd.Series) -> float:
    aligned = pd.concat([actual, predicted], axis=1).dropna()
    if aligned.empty:
        return np.nan
    return float((aligned.iloc[:, 0] - aligned.iloc[:, 1]).abs().mean())


model_frame = (
    lagged[["timestamp_utc", target_col, *lagged_feature_cols]].dropna().copy()
)

if len(model_frame) < 72:
    print(
        f"Only {len(model_frame)} complete lagged rows are available. "
        "Treat model diagnostics as a pipeline smoke test."
    )

model_frame["persistence_1h"] = features[target_col].shift(1).loc[model_frame.index]
model_frame["seasonal_naive_24h"] = (
    features[target_col].shift(24).loc[model_frame.index]
)

selected_lagged_features = (
    lagged_association_rank.head(5)["feature"].tolist()
    if len(lagged_association_rank)
    else []
)

split_index = int(len(model_frame) * 0.7)
train = model_frame.iloc[:split_index].copy()
test = model_frame.iloc[split_index:].copy()

results = {
    "test_rows": len(test),
    "persistence_1h_mae": mae(test[target_col], test["persistence_1h"]),
    "seasonal_naive_24h_mae": mae(test[target_col], test["seasonal_naive_24h"]),
}

if (
    selected_lagged_features
    and len(train) > len(selected_lagged_features) + 2
    and len(test)
):
    x_train = np.column_stack(
        [np.ones(len(train)), train[selected_lagged_features].to_numpy(dtype=float)]
    )
    y_train = train[target_col].to_numpy(dtype=float)
    beta, *_ = np.linalg.lstsq(x_train, y_train, rcond=None)

    x_test = np.column_stack(
        [np.ones(len(test)), test[selected_lagged_features].to_numpy(dtype=float)]
    )
    test["linear_lagged_feature_pred"] = x_test @ beta
    results["linear_lagged_feature_mae"] = mae(
        test[target_col], test["linear_lagged_feature_pred"]
    )
    results["linear_features"] = selected_lagged_features

pd.Series(results)

## Interpretation Notes

- High same-hour correlation does not prove forecast value.
- Lagged predictive signal is more realistic, but still incomplete without operational weather forecasts.
- A feature can be weak on average but important during extremes.
- The next serious run should repeat this over a full year or multiple years and stratify by season/extreme regimes.